# Main Steps

- Handle missing values
- Remove duplicates, none
- Convert data types
- Tranform various columns
- Derived columns
- Save the clean dataset

In [20]:
import pandas as pd
import numpy as np

In [21]:
df=pd.read_csv("../data/food_delivery_order_raw.csv")
df.head()

,Restaurant ID,Restaurant name,Subzone,City,Order ID,Order Placed At,Order Status,Delivery,Distance,Items in order,...,Rating,Review,Cancellation / Rejection reason,Restaurant compensation (Cancellation),Restaurant penalty (Rejection),KPT duration (minutes),Rider wait time (minutes),Order Ready Marked,Customer complaint tag,Customer ID
0,20320607,Swaad,Sector 4,Delhi NCR,6168884918,"11:38 PM, September 10 2024",Delivered,Zomato Delivery,3km,"1 x Grilled Chicken Jamaican Tender, 1 x Grill...",...,NaN,NaN,NaN,NaN,NaN,18.35,11.6,Correctly,NaN,5d6c2b96db963098bc69768bea504c8bf46106a8a5178e...
1,20320607,Swaad,Sector 4,Delhi NCR,6170707559,"11:34 PM, September 10 2024",Delivered,Zomato Delivery,2km,"1 x Peri Peri Fries, 1 x Fried Chicken Angara ...",...,NaN,NaN,NaN,NaN,NaN,16.95,3.6,Correctly,NaN,0781815deb4a10a574e9fee4fa0b86b074d4a0b36175d5...
2,20320607,Swaad,Sector 4,Delhi NCR,6169375019,"03:52 PM, September 10 2024",Delivered,Zomato Delivery,<1km,1 x Bone in Peri Peri Grilled Chicken,...,NaN,NaN,NaN,NaN,NaN,14.05,12.2,Correctly,NaN,f93362f5ce5382657482d164e368186bcec9c6225fd93d...
3,20320607,Swaad,Sector 4,Delhi NCR,6151677434,"03:45 PM, September 10 2024",Delivered,Zomato Delivery,2km,"1 x Fried Chicken Ghostbuster Tender, 1 x Anga...",...,4.0,NaN,NaN,NaN,NaN,19.00,3.3,Correctly,NaN,1ed226d1b8a5f7acee12fc1d6676558330a3b2b742af5d...
4,20320607,Swaad,Sector 4,Delhi NCR,6167540897,"03:04 PM, September 10 2024",Delivered,Zomato Delivery,2km,"1 x Peri Peri Krispers, 1 x Fried Chicken Anga...",...,NaN,NaN,NaN,NaN,NaN,15.97,1.0,Correctly,NaN,d21a2ac6ea06b31cc3288ab20c4ef2f292066c096f2c5f...


In [22]:
df.shape

(21321, 29)

In [23]:
df["Order ID"].duplicated().sum()
# if duplicates, then
# df=df.drop_duplicates()

np.int64(0)

In [24]:
df["Order Placed At"].dtype #string datatype

<StringDtype(na_value=nan)>

In [25]:
df["Order Placed At"] = pd.to_datetime(
    df["Order Placed At"],
    errors="coerce"
)
#if error then convert to NaT (Not a Time)

C:\Users\DELL\AppData\Local\Temp\ipykernel_23076\257338176.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Order Placed At"] = pd.to_datetime(


In [26]:
df["Order Placed At"].dtype
#us means microsecond, M8 is numpy's code for datetime64

dtype('<M8[us]')

In [27]:
#distance column has <1km and 3 km, convert to numerical column for analysis
df["Distance_km"] = (
    df["Distance"]
    .str.replace("km", "", regex=False)
    .str.replace("<1", "0.5", regex=False)
    .astype(float)
)
#regex=False means to find the exact km and replace it, instead of treating it as pattern

No need to change monetary columns and distance during transformation as they are already numerical fields and have float64 dt

In [28]:
#remove spaces from both ends
#select columns with string as dtype
text_columns = df.select_dtypes(include="object").columns

for col in text_columns:
    df[col] = df[col].str.strip()

C:\Users\DELL\AppData\Local\Temp\ipykernel_23076\1275317031.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(include="object").columns


Handling missing values, for cancellation, missing value might mean the order is not cancelled so it should be filled with zero

For columns such as ratings, reviews, complaints missing value should remain as it is

In [29]:
df.isnull().sum()

Restaurant ID                                             0
Restaurant name                                           0
Subzone                                                   0
City                                                      0
Order ID                                                  0
Order Placed At                                           0
Order Status                                              0
Delivery                                                  0
Distance                                                  0
Items in order                                            0
Instructions                                          20601
Discount construct                                     5498
Bill subtotal                                             0
Packaging charges                                         0
Restaurant discount (Promo)                               0
Restaurant discount (Flat offs, Freebies & others)        0
Gold discount                           

In [30]:
zero_fill_columns = [
    "Restaurant compensation (Cancellation)",
    "Restaurant penalty (Rejection)"
]

for col in zero_fill_columns:
    df[col] = df[col].fillna(0)

In [31]:
df.isnull().sum()

Restaurant ID                                             0
Restaurant name                                           0
Subzone                                                   0
City                                                      0
Order ID                                                  0
Order Placed At                                           0
Order Status                                              0
Delivery                                                  0
Distance                                                  0
Items in order                                            0
Instructions                                          20601
Discount construct                                     5498
Bill subtotal                                             0
Packaging charges                                         0
Restaurant discount (Promo)                               0
Restaurant discount (Flat offs, Freebies & others)        0
Gold discount                           

In [32]:
#make useful columns for anaylsis

df["Order Date"] = df["Order Placed At"].dt.date
df["Order Year"] = df["Order Placed At"].dt.year
df["Order Month"] = df["Order Placed At"].dt.month
df["Order Day"] = df["Order Placed At"].dt.day
df["Order Day of Week"] = df["Order Placed At"].dt.day_name()

In [33]:
#total discount column by combining all discount fields
discount_columns = [
    "Restaurant discount (Promo)",
    "Restaurant discount (Flat offs, Freebies & others)",
    "Gold discount",
    "Brand pack discount"
]

df["Total Discount"] = df[discount_columns].fillna(0).sum(axis=1)

In [34]:
#item count column
#fillna, replace NaN values with empty string so count doesnt have any problem
df["Item Count"] = (
    df["Items in order"]
    .fillna("")
    .str.count(",") + 1 #count no. of commas
)
# pizza 0
# pizza,burger 1
# so total items will be +1

df.loc[df["Items in order"].isna(), "Item Count"] = 0
#where items in order is null, set that to zero

In [35]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 21321 entries, 0 to 21320
Data columns (total 37 columns):
 #   Column                                              Non-Null Count  Dtype         
---  ------                                              --------------  -----         
 0   Restaurant ID                                       21321 non-null  int64         
 1   Restaurant name                                     21321 non-null  str           
 2   Subzone                                             21321 non-null  str           
 3   City                                                21321 non-null  str           
 4   Order ID                                            21321 non-null  int64         
 5   Order Placed At                                     21321 non-null  datetime64[us]
 6   Order Status                                        21321 non-null  str           
 7   Delivery                                            21321 non-null  str           
 8   Distance         

In [36]:
df.shape

(21321, 37)

In [37]:
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)
#columns only with missing values and then in descending order

Cancellation / Rejection reason    21135
Review                             21025
Customer complaint tag             20852
Instructions                       20601
Rating                             18830
Discount construct                  5498
KPT duration (minutes)               295
Rider wait time (minutes)            168
dtype: int64

In [38]:
#saving cleaned dataset
output_path = "../data/food_delivery_order_cleaned.csv"
df.to_csv(output_path, index=False)
#dont use the dataframe index as actual column
print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.


We still haven't split Items in Order in diff columns nor filled missing values for reasons. review. Because we might not need them during anaylsis or for data warehouse design, which can be decided at later stage